In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml kailash-align

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# M6 Ollama bootstrap — local LLMs, no API keys
# On Colab: installs + starts Ollama and pulls this lesson's models.
# Locally: verifies the daemon is running and models are pulled.
# ══════════════════════════════════════════════════════════════════
_LESSON_MODELS = ['qwen2.5:0.5b', 'llama3.2:3b']
import sys, os, time, shutil, subprocess
import httpx

_OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
_IN_COLAB = 'google.colab' in sys.modules

def _daemon_up(timeout_s=1.0):
    try:
        r = httpx.get(f'{_OLLAMA_BASE_URL}/api/tags', timeout=timeout_s)
        r.raise_for_status()
        return r.json()
    except Exception:
        return None

if _IN_COLAB and shutil.which('ollama') is None:
    print('[ollama] installing binary …')
    subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, check=True,
    )

if _IN_COLAB and _daemon_up() is None:
    print('[ollama] starting daemon …')
    subprocess.Popen(
        'nohup ollama serve > /tmp/ollama.log 2>&1 &',
        shell=True,
    )
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline and _daemon_up() is None:
        time.sleep(0.5)

_tags = _daemon_up(timeout_s=2.0)
if _tags is None:
    raise RuntimeError(
        f'Ollama daemon not reachable at {_OLLAMA_BASE_URL}. '
        'On Colab: re-run this cell. Locally: run `ollama serve`.'
    )

_have = {m.get('name','').split(':',1)[0] for m in _tags.get('models', [])}
for _model in _LESSON_MODELS:
    _fam = _model.split(':', 1)[0]
    if _fam in _have:
        continue
    if _IN_COLAB:
        print(f'[ollama] pulling {_model} (one-time) …')
        subprocess.run(['ollama', 'pull', _model], check=True)
    else:
        raise RuntimeError(
            f'Model {_model!r} not pulled. Run: ollama pull {_model}'
        )

print(
    '✓ Ollama ready at', _OLLAMA_BASE_URL,
    '— models:', _LESSON_MODELS,
)



In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp06
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


def create_visualizer():
    """Return a ModelVisualizer with the P2 experimental notice acknowledged.

    kailash-ml 2.2.2 emits ExperimentalWarning (a UserWarning) at
    ModelVisualizer construction; exercises run under warnings-as-errors,
    so the notice is acknowledged narrowly here — the ONE construction site
    every course file should use.
    """
    import warnings as _warnings

    from kailash_ml import ModelVisualizer
    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()

# ── SDK-internal deprecation nag filter (strict-gate hygiene) ────────────
# torch 2.12's ONNX exporter warns that dynamic_axes is "not recommended"
# under dynamo=True — a deprecation notice inside OnnxBridge's internals,
# not a correctness signal, and fatal under warnings-as-errors. Acknowledged
# here (kailash_helpers is imported by every shared.mlfpNN package, so every
# technique file gets it); message-matched, everything else still errors.
import warnings as _warnings

_warnings.filterwarnings(
    "ignore", message=r".*dynamic_axes.*not recommended.*", category=UserWarning
)
# matplotlib's Agg backend (forced in headless/script runs) nags
# "FigureCanvasAgg is non-interactive, and thus cannot be shown" on every
# plt.show() — a backend notice, not a correctness signal.
_warnings.filterwarnings(
    "ignore", message=r".*non-interactive.*cannot be shown.*", category=UserWarning
)

def hdb_storey_range_expr(column: str = "storey_range"):
    """Polars expression normalising HDB storey_range letter-O typos.

    The raw HDB resale file has letter-O typos ("O4 TO 06", "1O TO 12",
    "28 TO 3O"); read a letter O next to a digit as zero, leaving "TO"
    alone. Single home for the rule (P2): M2 ex_8 and M3 ex_5/07 both
    normalise at the use-site.
    """
    import polars as pl

    return (
        pl.col(column)
        .str.replace_all(r"\bO(\d)", "0${1}")
        .str.replace_all(r"(\d)O\b", "${1}0")
    )



# ── shared/mlfp06/ex_2.py ──
"""
Shared infrastructure for Exercise 2 — LLM Fine-Tuning.

Contains: SFT dataset loader (IMDB -> instruction pairs), LoRA/adapter
parameter-counting helpers, kailash-align config factory, base
hyperparameters, and output directory. Technique-specific classes
(LoRALayer, AdapterLayer, etc.) live in the per-technique files.
"""

import os
from pathlib import Path

import polars as pl
import torch


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
device = get_device()

# Output directory for all visualisation and training artifacts
REPO_ROOT = Path.cwd()
OUTPUT_DIR = REPO_ROOT / "outputs" / "ex2_finetuning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# SHARED HYPERPARAMETERS
# ════════════════════════════════════════════════════════════════════════

D_MODEL = 512  # synthetic demo hidden dim for from-scratch LoRA / adapter
LORA_RANK = 8  # default LoRA rank for the demos
LORA_ALPHA = 16.0  # typical alpha = 2 * rank
ADAPTER_BOTTLENECK = 64  # default adapter bottleneck

# ════════════════════════════════════════════════════════════════════════
# SFT DATA LOADING — IMDB instruction-response pairs
# ════════════════════════════════════════════════════════════════════════

DATA_DIR = REPO_ROOT / "data" / "mlfp06" / "imdb"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = DATA_DIR / "imdb_sft_2k.parquet"


def load_imdb_sft(
    n_rows: int = 2000, train_frac: float = 0.9
) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Load IMDB movie reviews and reformat as SFT instruction pairs.

    Returns:
        (full, train, eval) polars DataFrames with columns:
            instruction, response, text, label
    """
    if CACHE_FILE.exists():
        print(f"Loading cached IMDB SFT pairs from {CACHE_FILE}")
        sft_data = pl.read_parquet(CACHE_FILE)
    else:
        print("Downloading stanfordnlp/imdb from HuggingFace (first run)...")
        from datasets import load_dataset

        ds = load_dataset("stanfordnlp/imdb", split="train")
        ds = ds.shuffle(seed=42).select(range(min(n_rows, len(ds))))

        label_names = {0: "negative", 1: "positive"}
        rows = []
        for row in ds:
            review = row["text"][:1500]
            sentiment = label_names[row["label"]]
            rows.append(
                {
                    "instruction": (
                        "Classify the sentiment of the following movie review as "
                        "either 'positive' or 'negative', then briefly justify "
                        f"your answer.\n\nReview: {review}"
                    ),
                    "response": (
                        f"Sentiment: {sentiment}. The reviewer expresses a clearly "
                        f"{sentiment} reaction to the film."
                    ),
                    "text": review,
                    "label": sentiment,
                }
            )
        sft_data = pl.DataFrame(rows)
        sft_data.write_parquet(CACHE_FILE)
        print(f"Cached {sft_data.height} SFT pairs to {CACHE_FILE}")

    n_train = int(sft_data.height * train_frac)
    train_data = sft_data[:n_train]
    eval_data = sft_data[n_train:]
    print(
        f"IMDB SFT loaded: {sft_data.height} pairs "
        f"({train_data.height} train / {eval_data.height} eval)"
    )
    return sft_data, train_data, eval_data


# ════════════════════════════════════════════════════════════════════════
# PARAMETER COUNTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def count_lora_params(
    d_model: int, rank: int, num_modules: int = 1, num_layers: int = 1
) -> int:
    """LoRA trainable params = 2 * d * r per target module per layer.

    For d=512, r=8, modules=1, layers=1 -> 8,192 params.
    """
    return (d_model * rank + rank * d_model) * num_modules * num_layers


def count_adapter_params(d_model: int, bottleneck: int, num_layers: int = 1) -> int:
    """Adapter params per layer: down(d*b+b) + up(b*d+d) + layernorm(2*d).

    Matches the AdapterLayer in 02_adapter_from_scratch.py.
    """
    return (
        d_model * bottleneck + bottleneck + bottleneck * d_model + d_model + 2 * d_model
    ) * num_layers


def full_finetune_params(
    d_model: int, num_modules: int = 1, num_layers: int = 1
) -> int:
    """Full fine-tuning baseline: d^2 per module per layer."""
    return d_model * d_model * num_modules * num_layers


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ALIGN CONFIG FACTORY
# ════════════════════════════════════════════════════════════════════════


def get_base_model_name() -> str:
    """Return the HuggingFace repo id of the SFT base model.

    Resolution: ``SFT_BASE_MODEL`` env var → ``Qwen/Qwen2.5-0.5B-Instruct``.
    Note: Align/TRL training needs a HuggingFace repo id, NOT an Ollama tag —
    the Ollama-side ``OLLAMA_FT_BASE_MODEL`` is for inference of the
    fine-tuned model after GGUF export, not for training.
    """
    return os.environ.get("SFT_BASE_MODEL") or "Qwen/Qwen2.5-0.5B-Instruct"


def build_sft_config(
    base_model: str | None = None,
    lora_r: int = 16,
    lora_alpha: int = 32,
    num_epochs: int = 3,
    output_subdir: str = "sft_output",
):
    """Build a kailash-align AlignmentConfig for SFT+LoRA training.

    Imported lazily so technique files that do not call SFT are not
    forced to install the align extra at import time.

    kailash-align 0.6.0+ uses a composed AlignmentConfig: top-level
    method + base_model_id + experiment_dir, plus required LoRAConfig
    + SFTConfig + DPOConfig sub-configs. The DPOConfig is unused for
    SFT but the constructor still expects it (default values are fine).
    """
    from kailash_align import AlignmentConfig, DPOConfig, LoRAConfig, SFTConfig

    return AlignmentConfig(
        method="sft",
        base_model_id=base_model or get_base_model_name(),
        lora=LoRAConfig(
            rank=lora_r,
            alpha=lora_alpha,
            target_modules=("q_proj", "v_proj"),
            dropout=0.05,
        ),
        sft=SFTConfig(
            num_train_epochs=num_epochs,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            warmup_ratio=0.1,
            max_seq_length=512,
        ),
        dpo=DPOConfig(),
        experiment_dir=str(OUTPUT_DIR / output_subdir),
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP06 — Exercise 2.6: SFT with kailash-align AlignmentPipeline
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build an AlignmentConfig for SFT + LoRA (no raw transformers.Trainer)
#   - Run AlignmentPipeline.train() on the IMDB instruction dataset
#   - Register the trained adapter in AdapterRegistry
#   - Visualise SFT training loss + throughput
#   - Apply adapter-registry discipline to a Singapore e-commerce scenario
#
# PREREQUISITES: Exercises 2.1-2.5
# ESTIMATED TIME: ~40 min (training dominates)
#
# FRAMEWORK-FIRST: kailash-align, NOT raw transformers.Trainer. The
# pipeline wraps TRL SFTTrainer but adds a single typed config, adapter
# serialisation, and registry integration — the same pattern you used
# in MLFP03 for TrainingPipeline.
#
# TASKS:
#   1. THEORY: why AlignmentPipeline beats raw SFTTrainer
#   2. BUILD: AlignmentConfig for SFT + LoRA r=16
#   3. TRAIN: pipeline.train() on IMDB instruction pairs
#   4. VISUALISE: SFT training loss + throughput
#   5. APPLY: Singapore e-commerce adapter registry governance
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import matplotlib.pyplot as plt
import polars as pl
from dotenv import load_dotenv


load_dotenv()



## THEORY — Why AlignmentPipeline (Framework-First)


In [ ]:
# Raw transformers.Trainer + peft.get_peft_model loses three guarantees:
#   1. One typed AlignmentConfig (LoRA/SFT/DPO sub-configs) — note it only
#      checks target_modules is non-empty, not that the names exist
#   2. Structured adapter artefact lifecycle (not a bag of .bin files)
#   3. AdapterRegistry hand-off between training and serving
# Framework-first rule: Engine layer first, primitives only when the
# engine cannot express the behaviour.



## TASK 1 — Load the SFT dataset


In [ ]:
print("\n" + "=" * 70)
print("TASK 1: Load IMDB SFT instruction pairs")
print("=" * 70)

sft_data, train_data, eval_data = load_imdb_sft()
print(f"Train: {train_data.height} pairs")
print(f"Eval:  {eval_data.height} pairs")



### Checkpoint 1


In [ ]:
assert train_data.height > 0, "Task 1: train split should not be empty"
assert eval_data.height > 0, "Task 1: eval split should not be empty"
print("✓ Checkpoint 1 passed — SFT data loaded\n")



## TASK 2 — BUILD: AlignmentConfig via the shared factory


In [ ]:
print("=" * 70)
print("TASK 2: Build AlignmentConfig for SFT + LoRA r=16")
print("=" * 70)

# kailash-align 0.7.3 resumes unconditionally from any checkpoint found in
# the experiment dir — a crashed earlier run leaves a completed-looking
# checkpoint and every re-run "trains" zero steps. Retrain deterministically:
import shutil

shutil.rmtree(OUTPUT_DIR / "sft_output", ignore_errors=True)

# TODO: Read the base model name from the environment via get_base_model_name()
base_model = ____
# TODO: Build an AlignmentConfig via build_sft_config(base_model=base_model,
# lora_r=16, lora_alpha=32, num_epochs=3, output_subdir="sft_output")
config = ____

print(f"  Method:        {config.method}")
print(f"  Base model:    {config.base_model_id}")
print(f"  LoRA:          r={config.lora.rank}, alpha={config.lora.alpha}")
print(f"  Target modules: {config.lora.target_modules}")
print(f"  Epochs:        {config.sft.num_train_epochs}")
print(f"  Batch size:    {config.sft.per_device_train_batch_size}")
print(f"  Learning rate: {config.sft.learning_rate}")
print(f"  Output dir:    {config.experiment_dir}")



### Checkpoint 2


In [ ]:
assert config.method == "sft", "Task 2: method should be 'sft'"
assert config.lora.rank == 16, "Task 2: LoRA rank should be 16"
assert "q_proj" in config.lora.target_modules, "Task 2: q_proj must be a target module"
print("✓ Checkpoint 2 passed — AlignmentConfig built\n")



## TASK 3 — TRAIN: AlignmentPipeline + register adapter


Train the SFT adapter and register it. Returns metrics dict.


In [ ]:
# Training needs the base model from HuggingFace and is slow on CPU. There
# is deliberately NO "skip and pretend" mode: MLFP_SKIP_SFT_TRAIN=1 stops
# the exercise with an explanation instead of reporting a made-up loss.

print("=" * 70)
print("TASK 3: AlignmentPipeline.train() -> AdapterRegistry")
print("=" * 70)


async def run_sft_and_register() -> dict:
    import os

    if os.environ.get("MLFP_SKIP_SFT_TRAIN") == "1":
        raise RuntimeError(
            "MLFP_SKIP_SFT_TRAIN=1 is set, so SFT training was skipped and "
            "there is no training loss to report. Unset it to run Task 3."
        )
    from kailash_align import (
        AdapterRegistry,
        AdapterSignature,
        AlignmentPipeline,
    )

    # TODO: Instantiate AlignmentPipeline(config)
    pipeline = ____
    print("  Running SFT training (this may take several minutes)...")
    # TODO: SFT trains on the `text` column, but `text` in train_data is the
    # RAW review. Build a one-column polars frame whose `text` is
    # instruction + "\n\n" + response (pl.col(...) + ... .alias("text")).
    sft_frame = ____
    # TODO: kailash-align checks `dataset.column_names`, which a polars
    # DataFrame does not have. Convert: Dataset.from_dict(
    #   sft_frame.to_dict(as_series=False)) with `from datasets import Dataset`.
    from datasets import Dataset

    train_data_hf = ____
    # TODO: await pipeline.train(train_data_hf, adapter_name="imdb_sentiment_sft_v1").
    # Note (kailash-align 0.7.3): `dataset` is positional, `adapter_name`
    # is REQUIRED, and there is NO eval_data parameter anymore.
    result = ____

    # result.training_metrics is the raw TRL TrainOutput.metrics dict:
    # train_loss + train_runtime + train_samples_per_second. There is NO
    # eval_loss (no eval dataset is configured under the new API).
    train_metrics = result.training_metrics
    metrics = {
        "train_loss": train_metrics.get("train_loss"),
        "train_runtime": train_metrics.get("train_runtime", 0),
        "train_samples_per_second": train_metrics.get(
            "train_samples_per_second", 0
        ),
        "adapter_path": result.adapter_path,
    }
    print(f"  Train loss:    {metrics['train_loss']:.4f}")
    print(f"  Train runtime: {metrics['train_runtime']:.0f}s")
    print(f"  Throughput:    {metrics['train_samples_per_second']:.1f} samples/s")

    # TODO: Instantiate registry = AdapterRegistry()
    registry = ____
    # TODO: Build an AdapterSignature with base_model_id=config.base_model_id,
    #   adapter_type="lora", training_method="sft"
    signature = ____
    # TODO: await registry.register_adapter(name=..., adapter_path=...,
    #   signature=..., training_metrics={"train_loss": ...}, tags=[...])
    # register_adapter returns an AdapterVersion (not a string).
    version = ____
    # TODO: Build a stable adapter_id string of the form
    #   f"{version.adapter_name}:v{version.version}"
    adapter_id = ____
    metrics["adapter_id"] = adapter_id
    print(f"  Registered as: {adapter_id}")

    return metrics


metrics  = await run_sft_and_register()



### Checkpoint 3


In [ ]:
assert metrics["train_loss"] is not None, "Task 3: training should produce a loss"
assert metrics["train_loss"] > 0, "Task 3: train loss should be positive"
print("✓ Checkpoint 3 passed — SFT training + registration complete\n")



## TASK 4 — VISUALISE: SFT training loss + throughput


In [ ]:
# The new train() API returns the TRL headline metrics (no eval split), so
# we plot the honest signal it provides: the final train loss alongside
# training throughput. We deliberately do NOT fabricate an eval curve —
# generalisation is measured by a separate held-out eval pass (Ex 3.4).

print("=" * 70)
print("TASK 4: Visualise SFT training loss + throughput")
print("=" * 70)

fig, (ax_loss, ax_tput) = plt.subplots(1, 2, figsize=(11, 5))

# Left: final train loss (single headline bar — train() returns the run
# mean, not a per-step curve; the per-step curve lives in the TRL logs).
ax_loss.bar(
    ["Train loss (run mean)"],
    [metrics["train_loss"]],
    color="steelblue",
    edgecolor="black",
)
ax_loss.annotate(
    f"{metrics['train_loss']:.3f}",
    xy=(0, metrics["train_loss"]),
    xytext=(0, 3),
    textcoords="offset points",
    ha="center",
)
ax_loss.set_ylabel("Cross-entropy loss")
ax_loss.set_title("SFT LoRA r=16 — train loss", fontweight="bold")
ax_loss.grid(True, axis="y", alpha=0.3)

# Right: throughput — the other honest signal the pipeline reports.
ax_tput.bar(
    ["samples/s"],
    [metrics["train_samples_per_second"]],
    color="seagreen",
    edgecolor="black",
)
ax_tput.annotate(
    f"{metrics['train_samples_per_second']:.1f}",
    xy=(0, metrics["train_samples_per_second"]),
    xytext=(0, 3),
    textcoords="offset points",
    ha="center",
)
ax_tput.set_ylabel("Samples / second")
ax_tput.set_title(
    f"Training throughput (runtime {metrics['train_runtime']:.0f}s)",
    fontweight="bold",
)
ax_tput.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
fname = OUTPUT_DIR / "ex2_sft_train_loss.png"
fname.unlink(missing_ok=True)  # so the checkpoint cannot pass on a stale file
plt.savefig(fname, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {fname}")

print(f"\n  Final train loss: {metrics['train_loss']:.3f}")
print("  Healthy SFT runs show train_loss falling across epochs; measure")
print("  generalisation with a separate held-out eval pass (Exercise 3.4).")



### Checkpoint 4


In [ ]:
assert fname.exists(), "Task 4: loss plot should exist"
print("✓ Checkpoint 4 passed — SFT train loss visualised\n")



## TASK 5 — APPLY: Singapore e-commerce adapter registry governance


In [ ]:
# SCENARIO: A Singapore e-commerce platform runs an LLM-powered
# customer service assistant.  Over 18 months, the team has trained
# 37 different LoRA adapters on top of the same 7B base: returns,
# refunds, promotions, shipping, loyalty tiers, billing, KYC, and so
# on.  Each adapter was trained by a different on-call engineer with
# no shared record of "which adapter is live in production right now".
#
# PROBLEM: when a shopper complains that "the chatbot said shipping
# is free above S$50 but it charged me", the team cannot trace which
# adapter produced the answer, which dataset it was trained on, or
# which eval metrics it passed at training time.  Customer trust
# erodes; the promotion team cannot roll back a bad adapter because
# they don't know which version is deployed.
#
# GOVERNANCE FIX: every trained adapter MUST go through AdapterRegistry
# at training time with:
#   - name: human-readable identifier (e.g. "shipping_v3")
#   - base_model: exact base checkpoint SHA
#   - method: sft_lora / dpo_lora / adapter / ...
#   - metrics: train loss, eval loss, business-level eval (CSAT proxy)
#   - tags: domain (shipping), audience (retail), legal-reviewed (yes)
#
# Downstream serving pulls by registry name + metric threshold, never
# by raw .bin filename.  Deployment becomes auditable: every live
# adapter has a registry row that links back to the training run.
#
# BUSINESS IMPACT (illustrative figures):
#   - Audit: the compliance team can trace any customer-facing LLM
#     response to a registered adapter + training run + eval report —
#     the evidence a regulator's technology-risk review asks for.
#   - Rollback: a bad adapter can be rolled back in minutes by
#     pointing the serving layer at the previous registered version.
#     Previous rollback took ~4 hours and required redeploying the
#     serving image.
#   - Retraining cost avoided: the registry prevents duplicate runs
#     of the same fine-tune.  Engineers used to retrain ~4 adapters
#     per quarter because they could not find the previous adapter
#     files.  At ~S$600/training run + ~1 day engineering time, that
#     is ~S$2,400 + 4 engineer-days per quarter avoided.
#
# ANNUAL BENEFIT: computed below from these assumptions (~S$21K in direct
# cost) plus audit traceability and faster incident response.
#
# THE RULE: no adapter ships to production without a registry entry.
# Treat AdapterRegistry the way you treat ModelRegistry in MLFP03 —
# the single source of truth for which artefact runs where.

print("Singapore e-commerce adapter governance:")
annual_retraining_saving = 4 * 4 * 600  # 4 quarters * 4 runs/quarter * S$600
annual_engineer_hours_saved = 4 * 4 * 8  # 4 * 4 * 1 day
engineer_hourly_sgd = 90
annual_engineer_saving = annual_engineer_hours_saved * engineer_hourly_sgd
total_annual = annual_retraining_saving + annual_engineer_saving
print(f"  Retraining runs avoided / year:      {4 * 4}")
print(f"  Annual retraining cost avoided:      S${annual_retraining_saving:,}")
print(f"  Engineer hours saved / year:         {annual_engineer_hours_saved}")
print(f"  Annual engineer time saving:         S${annual_engineer_saving:,}")
print(f"  Total direct annual saving:          S${total_annual:,}")
print("  Plus: audit traceability + rollback SLA (not quantified here)")
print("  Recommended: AdapterRegistry mandatory for all SFT runs")



### Checkpoint 5


In [ ]:
assert total_annual > 0, "Task 5: governance should deliver positive ROI"
print("✓ Checkpoint 5 passed — e-commerce governance analysed\n")



## REFLECTION


[x] Built an AlignmentConfig for SFT + LoRA via kailash-align
      (framework-first: no raw transformers.Trainer)
  [x] Ran AlignmentPipeline.train() end-to-end on IMDB SFT data
  [x] Registered the trained adapter in AdapterRegistry with metrics
  [x] Visualised SFT train loss + throughput (the honest signals the
      0.7.3 train() API reports — no fabricated eval curve)
  [x] Applied adapter-registry governance to a Singapore e-commerce
      scenario (illustrative direct saving + audit traceability)

  KEY INSIGHT: SFT is the first rung of the alignment ladder.
  AlignmentPipeline + AdapterRegistry give you the versioning and
  audit trail you need before you layer DPO, GRPO, or RLHF on top.

  Next exercise (Exercise 3) moves from "learn the right response"
  (SFT) to "learn which response is PREFERRED" (DPO). Preference
  data replaces instruction pairs as the training signal.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

